In [ ]:
import csv
import json

with open("data/appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Total rows read:", len(rows))


def parse_row(row):
    parsed = row.copy()
    parsed["age"] = int(parsed["age"])
    parsed["fee_inr"] = int(parsed["fee_inr"])
    parsed["prior_no_shows"] = int(parsed["prior_no_shows"])
    parsed["distance_km"] = float(parsed["distance_km"])
    return parsed


accepted = []
rejected = []

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted rows: 988
Rejected rows: 32


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/appointments.csv")

print("Raw shape:", df.shape)
print("\nDataFrame info:")
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Raw distinct booking_channel values:", df["booking_channel"].nunique())

df = df.drop_duplicates().copy()
print("\nRow count after (a) dropping duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Row count after (b) cleaning booking_channel:", len(df))
print("Cleaned booking_channel values:", sorted(df["booking_channel"].dropna().unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("Row count after (c) converting dates:", len(df))
print("Unreadable appt_date count:", df["appt_date"].isnull().sum())

df = df.dropna(subset=["appt_date", "status"]).copy()
print("Row count after (d) dropping missing appt_date or status:", len(df))

Raw shape: (1020, 14)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values per column:
 appointment_id      0
pa

In [ ]:
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid age rows set to missing:", invalid_age_mask.sum())
df.loc[invalid_age_mask, "age"] = np.nan
print("Total missing ages before imputation:", df["age"].isnull().sum())

age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Median age: {age_median}, Median distance_km: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower}, upper={upper}, capped={capped}")

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee_inr after capping and rounding:", df["fee_inr"].max())

Invalid age rows set to missing: 5
Total missing ages before imputation: 17
Median age: 48.0, Median distance_km: 8.25
distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower=-7.9375, upper=25.5625, capped=26
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower=68.75, upper=1218.75, capped=3
Largest fee_inr after capping and rounding: 1219


In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"No-shows: {df['no_show'].sum()} / {len(df)} ({100 * df['no_show'].mean():.1f}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

col_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

lead_mask = X[:, 3] > 14
print(f"lead_days > 14: {lead_mask.sum()} rows ({100 * lead_mask.mean():.1f}%)")

dist_col = X[:, 1]
z_dist = (dist_col - np.mean(dist_col)) / np.std(dist_col)
print("Rows with |z| > 2 for distance_km:", (np.abs(z_dist) > 2).sum())

lead_days range: 1 to 45
No-shows: 218 / 990 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
lead_days > 14: 296 rows (29.9%)
Rows with |z| > 2 for distance_km: 53


In [ ]:
import requests

clinics_df = pd.read_csv("data/clinics.csv")
temps = []

for _, row in clinics_df.iterrows():
    temp_val = None
    reason = ""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }

    for attempt in range(2):
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                payload = resp.json()
                hourly = payload.get("hourly", {})
                readings = hourly.get("temperature_2m", [])
                if readings:
                    temp_val = readings[0]
                    break
                else:
                    reason = "Missing hourly temperature_2m in JSON"
            else:
                reason = f"HTTP status {resp.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)

    if temp_val is None:
        print(f"Failed for {row['clinic_name']}: {reason}")
    else:
        print(f"{row['clinic_name']} ({row['city']}): {temp_val} C")

    temps.append(temp_val)

clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)
print("\nSaved clinics_enriched.csv with shape:", clinics_df.shape)

Kothrud Family Clinic (Pune): 19.2 C
Hadapsar Care Centre (Pune): 19.5 C
Sitabuldi Health Point (Nagpur): 18.6 C
Malviya Nagar Clinic (Jaipur): 14.4 C
Vaishali Nagar Clinic (Jaipur): 14.4 C
Edappally Medical Centre (Kochi): 27.9 C
Vijay Nagar Clinic (Indore): 16.6 C
Gomti Nagar Care Point (Lucknow): 14.0 C

Saved clinics_enriched.csv with shape: (8, 7)


In [ ]:
export_cols = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
df = df[export_cols]

print("appointment_id is unique:", df["appointment_id"].is_unique)
print("Distinct appointment_id count:", df["appointment_id"].nunique())
print("Total missing values across 16 columns:", df.isnull().sum().sum())

df.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv with shape:", df.shape)

clinics_enriched = pd.read_csv("clinics_enriched.csv")
sql_lines = []

for _, r in clinics_enriched.iterrows():
    temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{r['temp_c']}"
    stmt = (
        f"INSERT INTO clinics VALUES ("
        f"'{r['clinic_id']}','{r['clinic_name']}','{r['city']}','{r['tier']}',"
        f"{r['latitude']},{r['longitude']},{temp_sql});"
    )
    sql_lines.append(stmt)

for _, r in df.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
        f"'{r['booked_on']}','{r['appt_date']}',{int(r['age'])},'{r['gender']}',"
        f"{r['distance_km']},{int(r['fee_inr'])},'{r['booking_channel']}',"
        f"'{r['visit_type']}','{r['reminder_sent']}',{int(r['prior_no_shows'])},"
        f"'{r['status']}',{int(r['lead_days'])},{int(r['no_show'])});"
    )
    sql_lines.append(stmt)

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines) + "\n")

print("Saved load_data.sql with total INSERT statements:", len(sql_lines))

appointment_id is unique: True
Distinct appointment_id count: 990
Total missing values across 16 columns: 0
Saved appointments_clean.csv with shape: (990, 16)
Saved load_data.sql with total INSERT statements: 998


In [ ]:
appts_clean = pd.read_csv("appointments_clean.csv")
clinics_enr = pd.read_csv("clinics_enriched.csv")

clinic_summary = (
    appts_clean.merge(clinics_enr[["clinic_id", "clinic_name", "city"]], on="clinic_id")
    .groupby(["clinic_name", "city"], as_index=False)
    .agg(appointments=("appointment_id", "count"), no_shows=("no_show", "sum"))
)
clinic_summary["no_show_pct"] = (100.0 * clinic_summary["no_shows"] / clinic_summary["appointments"]).round(1)
clinic_summary = clinic_summary.sort_values("no_show_pct", ascending=False)
print(clinic_summary.to_string(index=False))

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
